# Ingeniería de variables y partición temporal

En la notebook `02_preprocesamiento.ipynb` construimos una tabla integrada con una fila por pozo y mes (`data/processed/dataset_modelado.csv`), que incluye la producción de petróleo del mes siguiente como variable objetivo.

En esta notebook transformamos esa tabla en los datasets que vamos a usar para entrenar y evaluar los modelos de regresión y clasificación. Para eso:

1. validamos la tabla y la variable objetivo de regresión;
2. construimos las variables predictoras (*features*) a partir del historial de producción, las características del pozo y la fracturación;
3. definimos la variable objetivo de clasificación;
4. dividimos los datos en entrenamiento, validación y test de forma cronológica;
5. revisamos que ninguna variable incorpore información del futuro (fuga de información o *data leakage*);
6. exportamos los datasets y documentamos las decisiones.

En esta etapa no entrenamos modelos.

**Definición del problema.** Para cada pozo `i` y mes `t`, queremos predecir la producción de petróleo del mes calendario siguiente:

> y(i, t) = producción de petróleo del pozo i en el mes t + 1

Suponemos que la predicción se realiza **al cierre del mes t**: en ese momento ya se conoce la producción del mes t y de los meses anteriores, pero no la del mes t + 1. Todas las variables predictoras deben respetar ese corte.

**Entradas:** `data/processed/dataset_modelado.csv` (notebook 02) y `data/processed/fracturacion_limpio.csv` (notebook 01), que usamos para obtener la fecha de carga de cada fractura.

**Salidas:** `train_modelado.csv`, `validation_modelado.csv`, `test_modelado.csv`, `inferencia_modelado.csv` y `columnas_modelado.json`, en `data/processed/`.

### Importación de librerías

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

# Indicamos que se visualicen todas las columnas del dataframe
pd.set_option("display.max_columns", None)

### 1. Carga y validación del dataset de modelado

#### 1.1. Carga de los datos

In [2]:
ruta_processed = Path("../data/processed")
ruta_obt = ruta_processed / "dataset_modelado.csv"
ruta_fracturacion = ruta_processed / "fracturacion_limpio.csv"

for ruta in [ruta_obt, ruta_fracturacion]:
    if not ruta.exists():
        raise FileNotFoundError(
            f"No se encontró {ruta}. Primero hay que ejecutar las notebooks 01_exploracion_datos.ipynb y 02_preprocesamiento.ipynb."
        )

obt = pd.read_csv(
    ruta_obt,
    parse_dates=["fecha", "fecha_primera_fractura", "fecha_ultima_fractura"],
    low_memory=False,
)

obt = obt.sort_values(["idpozo", "fecha"]).reset_index(drop=True)

print(f"Filas: {len(obt):,}")
print("Columnas:", obt.shape[1])
print(f"Pozos: {obt['idpozo'].nunique():,}")
print("Período:", obt["fecha"].min().strftime("%Y-%m"), "a", obt["fecha"].max().strftime("%Y-%m"))
print(f"Filas con variable objetivo: {obt['target_disponible'].sum():,}")
print(f"Filas del período de modelado (2025 y 2026): {obt['periodo_modelado'].sum():,}")

Filas: 421,046
Columnas: 49
Pozos: 5,089
Período: 2006-01 a 2026-07
Filas con variable objetivo: 415,126
Filas del período de modelado (2025 y 2026): 88,610


Los valores coinciden con los informados en la sección 5.2 de la notebook 02.

#### 1.2. Validación de la variable objetivo de regresión

La variable objetivo `prod_pet_mes_siguiente` fue construida en la sección 4.2 de la notebook 02. Antes de usarla, verificamos de forma independiente que:

- corresponda a la producción del mismo pozo en el **mes calendario siguiente**, y no en la fila siguiente;
- falte exactamente cuando ese mes no existe (último mes del pozo o mes faltante en su serie);
- no tenga valores negativos.

Su unidad son metros cúbicos (m³), según la validación con la serie oficial de la sección 1.5 de la notebook 02.

In [3]:
assert not obt.duplicated(["idpozo", "fecha"]).any(), "Hay combinaciones de pozo y mes repetidas"

produccion_mes_siguiente = (
    obt[["idpozo", "fecha", "prod_pet"]]
    .assign(fecha=lambda tabla: tabla["fecha"] - pd.DateOffset(months=1))
    .rename(columns={"prod_pet": "target_verificado"})
)

verificacion = obt[["idpozo", "fecha", "prod_pet_mes_siguiente"]].merge(
    produccion_mes_siguiente, on=["idpozo", "fecha"], how="left", validate="one_to_one"
)

coincide_target = np.isclose(
    verificacion["prod_pet_mes_siguiente"], verificacion["target_verificado"], equal_nan=True
)

print("Filas donde el target coincide con la producción del mes calendario siguiente:", coincide_target.sum(), "de", len(obt))
print("Targets negativos:", (obt["prod_pet_mes_siguiente"] < 0).sum())
print("Targets faltantes:", obt["prod_pet_mes_siguiente"].isna().sum())

assert coincide_target.all(), "El target no coincide con la producción del mes siguiente"
assert (obt["target_disponible"] == obt["prod_pet_mes_siguiente"].notna()).all()
assert (obt["prod_pet_mes_siguiente"].dropna() >= 0).all()

Filas donde el target coincide con la producción del mes calendario siguiente: 421046 de 421046
Targets negativos: 0
Targets faltantes: 5920


La variable objetivo es correcta, por lo que la usamos tal como fue construida. Los valores iguales a cero se conservan, ya que representan meses reales sin producción de petróleo.

#### 1.3. Inventario de columnas

Clasificamos las 49 columnas de la tabla según la información que representan. Esta clasificación es el punto de partida para decidir qué columnas pueden usarse como variables predictoras.

In [4]:
inventario = {
    "Identificador": ["idpozo", "sigla"],
    "Temporal": [
        "fecha", "anio", "mes", "meses_desde_inicio", "meses_desde_registro_anterior",
        "inicio_censurado", "periodo_modelado",
    ],
    "Producción del mes": ["prod_pet", "prod_gas", "prod_agua", "iny_agua", "iny_gas", "tef"],
    "Característica fija del pozo": [
        "cuenca", "provincia", "areapermisoconcesion", "areayacimiento", "formacion",
        "sub_tipo_recurso", "clasificacion", "subclasificacion", "proyecto",
        "profundidad", "cota", "coordenadax", "coordenaday",
    ],
    "Característica operativa del mes": ["tipoestado", "tipoextraccion", "tipopozo", "empresa"],
    "Fracturación (por fecha del evento)": [
        "tiene_datos_fractura", "fractura_previa", "eventos_fractura_acum",
        "cantidad_fracturas_acum", "agua_inyectada_m3_acum", "arena_bombeada_tn_acum",
        "longitud_rama_horizontal_m_max", "presion_maxima_psi_max",
        "potencia_equipos_fractura_hp_max", "tipo_terminacion_ultima",
        "fecha_primera_fractura", "fecha_ultima_fractura", "meses_desde_ultima_fractura",
    ],
    "Control": ["prod_corregida", "tef_corregido", "target_disponible"],
    "Target de regresión": ["prod_pet_mes_siguiente"],
}

columnas_inventario = [columna for columnas in inventario.values() for columna in columnas]
assert sorted(columnas_inventario) == sorted(obt.columns), "Hay columnas sin clasificar o repetidas"

pd.DataFrame(
    [(grupo, len(columnas), ", ".join(columnas)) for grupo, columnas in inventario.items()],
    columns=["grupo", "cantidad", "columnas"],
)

,grupo,cantidad,columnas
0,Identificador,2,"idpozo, sigla"
1,Temporal,7,"fecha, anio, mes, meses_desde_inicio, meses_de..."
2,Producción del mes,6,"prod_pet, prod_gas, prod_agua, iny_agua, iny_g..."
3,Característica fija del pozo,13,"cuenca, provincia, areapermisoconcesion, areay..."
4,Característica operativa del mes,4,"tipoestado, tipoextraccion, tipopozo, empresa"
5,Fracturación (por fecha del evento),13,"tiene_datos_fractura, fractura_previa, eventos..."
6,Control,3,"prod_corregida, tef_corregido, target_disponible"
7,Target de regresión,1,prod_pet_mes_siguiente


Algunas observaciones que condicionan las siguientes secciones:

- **Fuga de información potencial:** el target (`prod_pet_mes_siguiente`) y su indicador de disponibilidad (`target_disponible`) nunca pueden ser variables predictoras. Además, `tiene_datos_fractura` indica si el pozo figura en el dataset de fracturación, lo que depende de cargas que pueden ser posteriores al mes de la fila. Las variables de fracturación también tienen un problema de disponibilidad, que tratamos en la sección 4.
- **Campos administrativos o de control:** `prod_corregida`, `tef_corregido`, `periodo_modelado` y `meses_desde_registro_anterior` describen el proceso de limpieza o la estructura de la serie, no el comportamiento del pozo.
- **Identificadores:** `idpozo` y `sigla` identifican cada pozo. Usarlos como variables permitiría al modelo memorizar pozos en lugar de aprender patrones generales.

### 2. Corte de predicción y población

#### 2.1. Mes de las variables y mes del objetivo

Cada fila de la tabla corresponde a un pozo `i` y un mes `t`. Distinguimos dos fechas:

| Concepto | Columna | Significado |
|---|---|---|
| Mes de corte | `fecha` | Último mes cuya información puede usarse. Las variables predictoras solo pueden usar datos de este mes o anteriores. |
| Mes del objetivo | `mes_objetivo` | Mes cuya producción se quiere predecir (`fecha` + 1 mes). |

Esta distinción es la base de la partición temporal de la sección 6: los períodos se definen según el mes del objetivo, para que ningún resultado de validación o test forme parte del entrenamiento.

In [5]:
obt["mes_objetivo"] = obt["fecha"] + pd.DateOffset(months=1)

display(obt[["idpozo", "fecha", "mes_objetivo", "prod_pet", "prod_pet_mes_siguiente"]].head(3))

,idpozo,fecha,mes_objetivo,prod_pet,prod_pet_mes_siguiente
0,3640,2006-02-01,2006-03-01,0.000,30.808
1,3640,2006-03-01,2006-04-01,30.808,76.609
2,3640,2006-04-01,2006-05-01,76.609,63.273


#### 2.2. Población

Incluimos **todos los tipos de pozo** (petrolíferos, gasíferos y otros). El tipo de pozo de cada mes (`tipopozo`) se incorpora como variable predictora, ya que condiciona fuertemente la producción de petróleo: en la sección 7.1 de la notebook 01 vimos que los pozos gasíferos tienen una mediana de producción de petróleo cercana a cero.

Siguiendo la conclusión de la notebook 01, los datasets de modelado incluyen solo los meses de corte de **2025 y 2026** (`periodo_modelado`). Sin embargo, las variables de historial se calculan sobre toda la serie desde 2006: por ejemplo, los rezagos de enero de 2025 necesitan la producción de 2024.

### 3. Variables de historial de producción

#### 3.1. Grilla de calendario

Para calcular rezagos y promedios móviles necesitamos que la posición de cada valor corresponda a un mes calendario. Si usáramos `shift()` o `rolling()` directamente sobre las filas de cada pozo, un pozo con meses faltantes tendría valores de meses no consecutivos tratados como si lo fueran. En la notebook 02 vimos que 704 pozos tienen al menos un mes faltante.

Por eso construimos una tabla auxiliar con una fila por mes calendario y una columna por pozo. Los meses sin registro quedan como faltantes. Esta tabla solo se usa para el cálculo: no agregamos filas nuevas al dataset ni completamos valores.

In [6]:
calendario = pd.date_range(obt["fecha"].min(), obt["fecha"].max(), freq="MS")

produccion_mensual = (
    obt.pivot(index="fecha", columns="idpozo", values="prod_pet")
    .reindex(calendario)
    .rename_axis(index="fecha")
)

print("Meses del calendario:", len(calendario))
print("Pozos:", produccion_mensual.shape[1])
print(f"Celdas con registro: {produccion_mensual.notna().sum().sum():,} (igual a la cantidad de filas de la tabla: {len(obt):,})")

Meses del calendario: 247
Pozos: 5089
Celdas con registro: 421,046 (igual a la cantidad de filas de la tabla: 421,046)


#### 3.2. Cálculo de las variables

A partir de la grilla calculamos las siguientes variables para cada pozo. Todas usan la producción del mes de corte `t` y de meses anteriores, nunca de meses posteriores:

| Variable | Cálculo |
|---|---|
| `prod_pet_lag_1`, `prod_pet_lag_2`, `prod_pet_lag_3` | Producción de 1, 2 y 3 meses antes del mes de corte. |
| `prod_pet_media_3m`, `prod_pet_media_6m` | Promedio de los últimos 3 y 6 meses, incluyendo el mes de corte. |
| `prod_pet_desvio_3m` | Desvío estándar de los últimos 3 meses. |
| `prod_pet_cambio_abs` | Diferencia entre la producción del mes de corte y la del mes anterior. |
| `prod_pet_cambio_rel` | Cambio relativo respecto del mes anterior. Solo se calcula si el mes anterior tuvo producción, para evitar divisiones por cero. |

Las ventanas se calculan a partir de los valores de cada mes de la ventana (el mes de corte y sus rezagos). Si falta cualquiera de ellos, el resultado queda como faltante, por lo que una ventana nunca mezcla meses separados por un hueco.

No usamos `rolling()` sobre la grilla porque su algoritmo acumula pequeños errores de redondeo a lo largo de toda la serie: la validación de la sección 3.3 detectó desvíos de 0,000002 m³ donde el valor exacto era 0. Calcular cada ventana con sus valores evita ese problema.

In [7]:
produccion_anterior = produccion_mensual.shift(1)

# Valores de cada mes de la ventana: el mes de corte (shift 0) y los meses anteriores
ventana_3m = [produccion_mensual.shift(k) for k in range(3)]
ventana_6m = [produccion_mensual.shift(k) for k in range(6)]

# np.std devuelve un valor faltante si falta cualquier mes de la ventana
desvio_3m = pd.DataFrame(
    np.std(np.stack(ventana_3m), axis=0, ddof=1),
    index=produccion_mensual.index,
    columns=produccion_mensual.columns,
)

features_historial = {
    "prod_pet_lag_1": produccion_mensual.shift(1),
    "prod_pet_lag_2": produccion_mensual.shift(2),
    "prod_pet_lag_3": produccion_mensual.shift(3),
    "prod_pet_media_3m": sum(ventana_3m) / 3,
    "prod_pet_media_6m": sum(ventana_6m) / 6,
    "prod_pet_desvio_3m": desvio_3m,
    "prod_pet_cambio_abs": produccion_mensual - produccion_anterior,
    "prod_pet_cambio_rel": (produccion_mensual - produccion_anterior) / produccion_anterior.where(produccion_anterior > 0),
}


def a_formato_largo(tabla, nombre):
    # Convierte la grilla (meses x pozos) en una columna con una fila por pozo y mes
    largo = tabla.reset_index().melt(id_vars="fecha", var_name="idpozo", value_name=nombre)
    return largo.astype({"idpozo": obt["idpozo"].dtype})


filas_antes = len(obt)

for nombre, tabla in features_historial.items():
    obt = obt.merge(a_formato_largo(tabla, nombre), on=["fecha", "idpozo"], how="left", validate="one_to_one")

assert len(obt) == filas_antes, "La incorporación de variables cambió la cantidad de filas"
assert obt["idpozo"].dtype == "int64", "El identificador del pozo cambió de tipo"

display(obt.loc[obt["idpozo"] == obt["idpozo"].iloc[0], ["fecha", "prod_pet"] + list(features_historial)].head(7))

,fecha,prod_pet,prod_pet_lag_1,prod_pet_lag_2,prod_pet_lag_3,prod_pet_media_3m,prod_pet_media_6m,prod_pet_desvio_3m,prod_pet_cambio_abs,prod_pet_cambio_rel
0,2006-02-01,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2006-03-01,30.808,0.000,NaN,NaN,NaN,NaN,NaN,30.808,NaN
2,2006-04-01,76.609,30.808,0.000,NaN,35.805667,NaN,38.548245,45.801,1.486659
3,2006-05-01,63.273,76.609,30.808,0.000,56.896667,NaN,23.556870,-13.336,-0.174079
4,2006-06-01,55.043,63.273,76.609,30.808,64.975000,NaN,10.883276,-8.230,-0.130071
5,2006-07-01,58.862,55.043,63.273,76.609,59.059333,47.432500,4.118547,3.819,0.069382
6,2006-08-01,33.241,58.862,55.043,63.273,49.048667,52.972667,13.822371,-25.621,-0.435272


Agregamos además dos variables que no necesitan la grilla:

- `prod_pet_acumulada`: producción acumulada del pozo hasta el mes de corte, incluido. Es una medida del agotamiento del pozo: a medida que un pozo produce, su producción tiende a declinar. Se calcula sobre los registros disponibles, por lo que los meses faltantes no suman producción.
- `prod_pet_por_dia`: producción del mes dividida por los días efectivos (`tef`). Permite distinguir un pozo que produjo poco porque estuvo parado de uno que produjo poco estando activo todo el mes. Solo se calcula cuando `tef` es mayor a 0.

In [8]:
obt["prod_pet_acumulada"] = obt.groupby("idpozo")["prod_pet"].cumsum()
obt["prod_pet_por_dia"] = obt["prod_pet"] / obt["tef"].where(obt["tef"] > 0)

print("Registros con producción pero sin días efectivos (prod_pet_por_dia faltante):", ((obt["tef"] == 0) & (obt["prod_pet"] > 0)).sum())

Registros con producción pero sin días efectivos (prod_pet_por_dia faltante): 264


#### 3.3. Validación de las variables de historial

Verificamos que las variables se hayan calculado correctamente comparándolas con un cálculo alternativo por filas: dentro de cada tramo de meses consecutivos del pozo, tomamos los valores de las filas anteriores con `shift()`. Este cálculo solo es válido cuando los meses anteriores son consecutivos, y no usa la grilla de calendario, por lo que sirve como control independiente.

Para esto identificamos, en cada fila, cuántos meses consecutivos con registro tiene el pozo hasta el mes de corte (`meses_consecutivos`). Si ese valor es al menos igual al tamaño de la ventana, la ventana por calendario y la ventana por filas deben coincidir; si es menor, la variable debe estar faltante.

In [9]:
nuevo_bloque = obt["meses_desde_registro_anterior"] != 1
bloque = nuevo_bloque.cumsum()
meses_consecutivos = obt.groupby(bloque).cumcount() + 1

validaciones_historial = {}

# Rezago de 1 mes
fila_anterior = obt.groupby("idpozo")["prod_pet"].shift(1)
con_mes_anterior = meses_consecutivos >= 2
validaciones_historial["Rezago 1 igual a la fila anterior cuando el mes anterior existe"] = np.allclose(
    obt.loc[con_mes_anterior, "prod_pet_lag_1"], fila_anterior[con_mes_anterior]
)
validaciones_historial["Rezago 1 faltante cuando el mes anterior no existe"] = obt.loc[~con_mes_anterior, "prod_pet_lag_1"].isna().all()

# Ventanas móviles: valores de las filas anteriores dentro de cada tramo de meses consecutivos
produccion_por_tramo = obt.groupby(bloque)["prod_pet"]
filas_3m = np.stack([produccion_por_tramo.shift(k) for k in range(3)])
filas_6m = np.stack([produccion_por_tramo.shift(k) for k in range(6)])

ventanas = {
    "prod_pet_media_3m": (3, filas_3m.mean(axis=0)),
    "prod_pet_media_6m": (6, filas_6m.mean(axis=0)),
    "prod_pet_desvio_3m": (3, filas_3m.std(axis=0, ddof=1)),
}

for nombre, (tamanio, por_filas) in ventanas.items():
    ventana_completa = meses_consecutivos >= tamanio
    validaciones_historial[f"{nombre} coincide con el cálculo por filas"] = np.allclose(
        obt.loc[ventana_completa, nombre], por_filas[ventana_completa.to_numpy()]
    )
    validaciones_historial[f"{nombre} faltante cuando la ventana tiene huecos o es incompleta"] = obt.loc[~ventana_completa, nombre].isna().all()

resultado_historial = pd.Series(validaciones_historial, name="cumple")
display(resultado_historial)

assert resultado_historial.all(), "Hay variables de historial mal calculadas"

Rezago 1 igual a la fila anterior cuando el mes anterior existe               True
Rezago 1 faltante cuando el mes anterior no existe                            True
prod_pet_media_3m coincide con el cálculo por filas                           True
prod_pet_media_3m faltante cuando la ventana tiene huecos o es incompleta     True
prod_pet_media_6m coincide con el cálculo por filas                           True
prod_pet_media_6m faltante cuando la ventana tiene huecos o es incompleta     True
prod_pet_desvio_3m coincide con el cálculo por filas                          True
prod_pet_desvio_3m faltante cuando la ventana tiene huecos o es incompleta    True
Name: cumple, dtype: bool

Todas las validaciones se cumplen. Por lo tanto, las variables de historial usan solo meses consecutivos anteriores o iguales al mes de corte.

### 4. Variables de fracturación

#### 4.1. Fecha del evento y fecha de disponibilidad

En la notebook 02 incorporamos la fracturación según la **fecha en que ocurrió** cada fractura. Sin embargo, en la sección 2.6 de esa notebook vimos que la información se cargó con demora: en la mitad de los casos, más de un año después del evento.

Esto plantea un problema de disponibilidad. Si al cierre de un mes una fractura ya había ocurrido pero todavía no estaba cargada, esa información no estaba disponible en ese momento. Distinguimos entonces dos fechas:

1. **Fecha del evento:** cuándo se realizó la fractura.
2. **Fecha de disponibilidad:** cuándo la información fue cargada, que es la más tardía entre el mes del evento y el mes de carga (`anio_carga` y `mes_carga`).

El dataset de la notebook 02 no conserva la fecha de carga. Por eso volvemos a leer el dataset de fracturación y recalculamos las variables usando la fecha de disponibilidad. Aplicamos los mismos criterios de limpieza que en las secciones 2.5 y 2.6 de la notebook 02: eliminamos los eventos cargados dos veces, usamos la fecha más tardía cuando el fin es anterior al inicio y tratamos las presiones iguales a 0 como faltantes.

In [10]:
fracturacion = pd.read_csv(
    ruta_fracturacion,
    parse_dates=["fecha_inicio_fractura", "fecha_fin_fractura"],
    low_memory=False,
)

columnas_evento = [
    "idpozo", "formacion_productiva", "fecha_inicio_fractura", "fecha_fin_fractura",
    "cantidad_fracturas", "longitud_rama_horizontal_m", "agua_inyectada_m3",
    "arena_bombeada_nacional_tn", "arena_bombeada_importada_tn",
    "presion_maxima_psi", "potencia_equipos_fractura_hp",
]

fracturacion = (
    fracturacion
    .sort_values(["anio_carga", "mes_carga", "id_base_fractura_adjiv"])
    .drop_duplicates(columnas_evento, keep="first")
    .reset_index(drop=True)
)

fracturacion["fecha_evento"] = fracturacion[["fecha_inicio_fractura", "fecha_fin_fractura"]].max(axis=1)
fracturacion.loc[fracturacion["presion_maxima_psi"] == 0, "presion_maxima_psi"] = np.nan

fracturacion["mes_evento"] = fracturacion["fecha_evento"].dt.to_period("M").dt.to_timestamp()
fracturacion["mes_carga_fractura"] = pd.to_datetime(
    fracturacion[["anio_carga", "mes_carga"]].rename(columns={"anio_carga": "year", "mes_carga": "month"}).assign(day=1)
)
fracturacion["mes_disponible"] = fracturacion[["mes_evento", "mes_carga_fractura"]].max(axis=1).astype(obt["fecha"].dtype)

print("Eventos de fracturación:", len(fracturacion))
print(f"Eventos cargados después del mes en que ocurrieron: {(fracturacion['mes_carga_fractura'] > fracturacion['mes_evento']).mean() * 100:.1f}%")

Eventos de fracturación: 4884
Eventos cargados después del mes en que ocurrieron: 99.0%


#### 4.2. Agregación por fecha de disponibilidad

Repetimos la agregación de la sección 3.5 de la notebook 02, pero ordenando los eventos por su fecha de disponibilidad. Para cada pozo acumulamos los eventos a medida que se van cargando:

- la cantidad de eventos y de fracturas, el agua inyectada y la arena bombeada se suman;
- la longitud de la rama horizontal, la presión máxima y la potencia toman el mayor valor disponible;
- la fecha y el tipo de terminación de la última fractura corresponden al evento más reciente **entre los ya cargados**, ya que un evento antiguo puede cargarse después que uno más nuevo.

Luego asociamos a cada mes de corte la última situación disponible con `pd.merge_asof`, que solo busca hacia atrás en el tiempo.

In [11]:
fracturacion_pozos = (
    fracturacion[fracturacion["idpozo"].isin(obt["idpozo"])]
    .sort_values(["idpozo", "mes_disponible", "fecha_evento"])
    .reset_index(drop=True)
)
fracturacion_pozos["arena_bombeada_tn"] = (
    fracturacion_pozos["arena_bombeada_nacional_tn"] + fracturacion_pozos["arena_bombeada_importada_tn"]
)

por_pozo = fracturacion_pozos.groupby("idpozo")

fracturacion_pozos["eventos_fractura_acum"] = por_pozo.cumcount() + 1
fracturacion_pozos["cantidad_fracturas_acum"] = por_pozo["cantidad_fracturas"].cumsum()
fracturacion_pozos["agua_inyectada_m3_acum"] = por_pozo["agua_inyectada_m3"].cumsum()
fracturacion_pozos["arena_bombeada_tn_acum"] = por_pozo["arena_bombeada_tn"].cumsum()
fracturacion_pozos["longitud_rama_horizontal_m_max"] = por_pozo["longitud_rama_horizontal_m"].cummax()
fracturacion_pozos["presion_maxima_psi_max"] = por_pozo["presion_maxima_psi"].cummax()
fracturacion_pozos["potencia_equipos_fractura_hp_max"] = por_pozo["potencia_equipos_fractura_hp"].cummax()
fracturacion_pozos["fecha_ultima_fractura"] = por_pozo["fecha_evento"].cummax()

# Tipo de terminación del evento más reciente entre los ya cargados
es_ultimo_evento = fracturacion_pozos["fecha_evento"] == fracturacion_pozos["fecha_ultima_fractura"]
fracturacion_pozos["tipo_terminacion_ultima"] = fracturacion_pozos["tipo_terminacion"].where(es_ultimo_evento)

# cummax deja faltantes cuando el dato del evento no fue informado:
# completamos con el valor disponible hasta el evento anterior del mismo pozo
columnas_completar = ["presion_maxima_psi_max", "potencia_equipos_fractura_hp_max", "tipo_terminacion_ultima"]
fracturacion_pozos[columnas_completar] = fracturacion_pozos.groupby("idpozo")[columnas_completar].ffill()

columnas_fractura = [
    "eventos_fractura_acum", "cantidad_fracturas_acum", "agua_inyectada_m3_acum",
    "arena_bombeada_tn_acum", "longitud_rama_horizontal_m_max", "presion_maxima_psi_max",
    "potencia_equipos_fractura_hp_max", "tipo_terminacion_ultima",
]

# Una fila por pozo y mes de disponibilidad: la situación acumulada al final de ese mes
fracturacion_disponible = (
    fracturacion_pozos
    .groupby(["idpozo", "mes_disponible"])
    .tail(1)[["idpozo", "mes_disponible", "fecha_ultima_fractura"] + columnas_fractura]
)

print("Eventos de pozos con producción:", len(fracturacion_pozos))
print("Filas por pozo y mes de disponibilidad:", len(fracturacion_disponible))

Eventos de pozos con producción: 3903
Filas por pozo y mes de disponibilidad: 3854


Antes de incorporar las nuevas variables, renombramos las versiones de la notebook 02 con el sufijo `_evento`. No las usamos como variables principales, pero las conservamos como un conjunto alternativo para un análisis de sensibilidad: comparar el desempeño de los modelos con ambas versiones permite medir el efecto de la demora de carga.

In [12]:
columnas_fractura_obt = columnas_fractura + ["fractura_previa", "meses_desde_ultima_fractura"]
obt = obt.rename(columns={columna: f"{columna}_evento" for columna in columnas_fractura_obt})
obt = obt.rename(columns={
    "fecha_primera_fractura": "fecha_primera_fractura_evento",
    "fecha_ultima_fractura": "fecha_ultima_fractura_evento",
})

filas_antes = len(obt)

obt = pd.merge_asof(
    obt.sort_values("fecha"),
    fracturacion_disponible.sort_values("mes_disponible"),
    left_on="fecha",
    right_on="mes_disponible",
    by="idpozo",
    direction="backward",
)
obt = obt.sort_values(["idpozo", "fecha"]).reset_index(drop=True)

obt["fractura_previa"] = obt["eventos_fractura_acum"].notna()
obt["meses_desde_ultima_fractura"] = (
    (obt["fecha"].dt.year - obt["fecha_ultima_fractura"].dt.year) * 12
    + (obt["fecha"].dt.month - obt["fecha_ultima_fractura"].dt.month)
)

assert len(obt) == filas_antes, "La incorporación de la fracturación cambió la cantidad de filas"
assert (obt["mes_disponible"].dropna() <= obt.loc[obt["mes_disponible"].notna(), "fecha"]).all(), "Hay fracturas no disponibles al mes de corte"

print("Filas antes:", filas_antes, "| Filas después:", len(obt))

Filas antes: 421046 | Filas después: 421046


Validamos la relación entre ambas versiones: como la fecha de disponibilidad nunca es anterior a la del evento, la versión por disponibilidad debe tener siempre la misma cantidad de eventos o menos.

In [13]:
eventos_disponibles = obt["eventos_fractura_acum"].fillna(0)
eventos_por_fecha = obt["eventos_fractura_acum_evento"].fillna(0)

assert (eventos_disponibles <= eventos_por_fecha).all(), "La versión por disponibilidad tiene eventos que la versión por fecha no tiene"

periodo = obt["periodo_modelado"]
difiere = eventos_disponibles != eventos_por_fecha

print(f"Filas del período de modelado donde ambas versiones difieren: {difiere[periodo].sum():,} ({difiere[periodo].mean() * 100:.1f}%)")
print(f"Filas del período de modelado con fractura previa (por fecha del evento): {obt.loc[periodo, 'fractura_previa_evento'].mean() * 100:.1f}%")
print(f"Filas del período de modelado con fractura disponible (por fecha de carga): {obt.loc[periodo, 'fractura_previa'].mean() * 100:.1f}%")

Filas del período de modelado donde ambas versiones difieren: 3,215 (3.6%)
Filas del período de modelado con fractura previa (por fecha del evento): 75.7%
Filas del período de modelado con fractura disponible (por fecha de carga): 72.1%


En el período de modelado, ambas versiones difieren en una minoría de filas: son los meses en los que una fractura ya había ocurrido pero todavía no estaba cargada. Para esos meses, la versión principal (por fecha de disponibilidad) refleja correctamente la información que se tenía en ese momento.

Por la misma razón, no usamos `tiene_datos_fractura` como variable: indica si el pozo aparece en algún momento en el dataset de fracturación, incluso por cargas posteriores al mes de corte. En su lugar, `fractura_previa` indica si había alguna fractura disponible al cierre de ese mes.

### 5. Variables temporales y características del pozo

#### 5.1. Antigüedad del pozo

El dataset no tiene una fecha confiable de puesta en marcha de cada pozo: en la sección 3.3 de la notebook 02 vimos que muchas fechas de terminación son del año 1900 o posteriores al inicio de la producción.

Por eso usamos `meses_desde_inicio`, que mide la **antigüedad del historial de producción observado**: los meses transcurridos desde el primer registro del pozo en el dataset. En la sección 4.1 de la notebook 02 verificamos que el año del primer registro coincide con el año de primera producción del padrón oficial, por lo que esta variable es una buena aproximación de la antigüedad productiva del pozo.

La excepción son los pozos con `inicio_censurado`, que ya producían en enero de 2006, el primer mes del dataset. Para ellos, la antigüedad real es mayor que `meses_desde_inicio`. Por eso incluimos `inicio_censurado` como variable: le indica al modelo que, en esos pozos, el historial observado no empieza con el inicio de la producción.

In [14]:
periodo = obt["periodo_modelado"]

print("Meses desde el inicio en el período de modelado:")
display(obt.loc[periodo, "meses_desde_inicio"].describe())
print(f"Filas del período de modelado con inicio censurado: {obt.loc[periodo, 'inicio_censurado'].mean() * 100:.1f}%")

Meses desde el inicio en el período de modelado:


count    88610.000000
mean        81.150288
std         59.129166
min          0.000000
25%         30.000000
50%         78.000000
75%        120.000000
max        246.000000
Name: meses_desde_inicio, dtype: float64

Filas del período de modelado con inicio censurado: 4.2%


#### 5.2. Mes y año calendario

Incluimos el mes calendario (`mes`) como variable, ya que puede reflejar efectos estacionales, por ejemplo la cantidad de días del mes.

No incluimos el año (`anio`). El entrenamiento usa casi únicamente meses de 2025 y la validación y el test meses de 2026, por lo que el modelo nunca vería en entrenamiento los valores de año que tiene que predecir. Un modelo basado en árboles trataría 2026 como si fuera 2025, y un modelo lineal extrapolaría una tendencia que no aprendió.

#### 5.3. Características del pozo y del mes

Seleccionamos las siguientes variables de la tabla de la notebook 02:

- **Características fijas:** `profundidad`, `cota`, `coordenadax`, `coordenaday`, `cuenca`, `formacion`, `sub_tipo_recurso`, `clasificacion`, `subclasificacion` y `proyecto`. En las secciones 3.5 a 3.7 de la notebook 01 vimos que la cuenca, la formación y la profundidad están relacionadas con la producción.
- **Características del mes:** `tipopozo`, `tipoestado`, `tipoextraccion`, `empresa`, `prod_gas`, `prod_agua` y `tef`. Describen la situación del pozo en el mes de corte, por lo que están disponibles al cierre de ese mes.

Excluimos:

- `provincia`, `areapermisoconcesion` y `areayacimiento`: repiten la ubicación que ya aportan las coordenadas y la cuenca, y las áreas tienen más de 100 categorías cada una, por lo que facilitarían que el modelo memorice zonas particulares;
- `iny_agua` e `iny_gas`: son distintas de 0 en menos del 1% de los registros del período de modelado, por lo que prácticamente no aportan información.

In [15]:
print(f"Registros del período de modelado con inyección de agua: {(obt.loc[periodo, 'iny_agua'] > 0).mean() * 100:.2f}%")
print(f"Registros del período de modelado con inyección de gas: {(obt.loc[periodo, 'iny_gas'] > 0).mean() * 100:.2f}%")
print("Categorías de areapermisoconcesion:", obt["areapermisoconcesion"].nunique())
print("Categorías de areayacimiento:", obt["areayacimiento"].nunique())
print("Categorías de empresa en el período de modelado:", obt.loc[periodo, "empresa"].nunique())

Registros del período de modelado con inyección de agua: 0.31%
Registros del período de modelado con inyección de gas: 0.39%
Categorías de areapermisoconcesion: 117
Categorías de areayacimiento: 156
Categorías de empresa en el período de modelado: 35


#### 5.4. Lista de variables predictoras

Reunimos las variables seleccionadas en listas, separadas según su tipo. Estas listas se exportan junto con los datasets para que las notebooks de modelado usen exactamente las mismas variables.

In [16]:
features_numericas = [
    # Historial de producción
    "prod_pet", "prod_pet_lag_1", "prod_pet_lag_2", "prod_pet_lag_3",
    "prod_pet_media_3m", "prod_pet_media_6m", "prod_pet_desvio_3m",
    "prod_pet_cambio_abs", "prod_pet_cambio_rel", "prod_pet_acumulada", "prod_pet_por_dia",
    # Producción y operación del mes
    "prod_gas", "prod_agua", "tef",
    # Características fijas del pozo
    "profundidad", "cota", "coordenadax", "coordenaday",
    # Temporales
    "mes", "meses_desde_inicio",
    # Fracturación disponible al mes de corte
    "eventos_fractura_acum", "cantidad_fracturas_acum", "agua_inyectada_m3_acum",
    "arena_bombeada_tn_acum", "longitud_rama_horizontal_m_max", "presion_maxima_psi_max",
    "potencia_equipos_fractura_hp_max", "meses_desde_ultima_fractura",
]

features_booleanas = ["inicio_censurado", "fractura_previa"]

features_categoricas = [
    "tipopozo", "tipoestado", "tipoextraccion", "empresa",
    "cuenca", "formacion", "sub_tipo_recurso", "clasificacion", "subclasificacion", "proyecto",
    "tipo_terminacion_ultima",
]

features = features_numericas + features_booleanas + features_categoricas

# Variables de fracturación por fecha del evento, para análisis de sensibilidad
features_sensibilidad = [f"{columna}_evento" for columna in columnas_fractura_obt]

print("Variables numéricas:", len(features_numericas))
print("Variables booleanas:", len(features_booleanas))
print("Variables categóricas:", len(features_categoricas))
print("Total de variables predictoras:", len(features))

Variables numéricas: 28
Variables booleanas: 2
Variables categóricas: 11
Total de variables predictoras: 41


### 6. Partición temporal

#### 6.1. Definición de los períodos

Como el objetivo es predecir meses futuros, una partición aleatoria no es adecuada: el modelo se entrenaría con meses posteriores a los que se usan para evaluarlo. Por eso dividimos los datos de forma cronológica.

Las filas disponibles para entrenar y evaluar son las del período de modelado que tienen variable objetivo: meses de corte de enero de 2025 a junio de 2026, es decir, meses del objetivo de febrero de 2025 a julio de 2026.

Definimos los períodos según el **mes del objetivo**:

| Partición | Meses del objetivo | Meses de corte | Uso |
|---|---|---|---|
| Entrenamiento | febrero a diciembre de 2025 | enero a noviembre de 2025 | Ajustar los modelos. |
| Validación | enero a marzo de 2026 | diciembre de 2025 a febrero de 2026 | Optimizar hiperparámetros y comparar modelos. |
| Test | abril a julio de 2026 | marzo a junio de 2026 | Evaluación final, una sola vez. |

Elegimos estos límites por los siguientes motivos:

- el test corresponde a los meses más recientes, que son los más parecidos a la situación en la que se usaría el modelo;
- la validación tiene tres meses, suficientes para comparar modelos sin reducir demasiado el entrenamiento;
- el entrenamiento conserva 11 de los 18 meses, para que el modelo vea casi un año completo de comportamiento;
- los límites coinciden con el cambio de año, lo que facilita su interpretación.

Como los períodos se definen por el mes del objetivo, ningún resultado de validación o test forma parte del entrenamiento. Las variables de una fila de validación sí pueden usar la producción de diciembre de 2025, que es el último objetivo del entrenamiento: esa información ya era conocida al momento de predecir, por lo que no constituye una fuga.

In [17]:
limites_particion = {
    "train": ("2025-02-01", "2025-12-01"),
    "validation": ("2026-01-01", "2026-03-01"),
    "test": ("2026-04-01", "2026-07-01"),
}

modelado = obt[obt["periodo_modelado"]].copy()

modelado["particion"] = pd.NA
for particion, (desde, hasta) in limites_particion.items():
    en_periodo = modelado["mes_objetivo"].between(desde, hasta)
    modelado.loc[en_periodo & modelado["target_disponible"], "particion"] = particion

sin_target = ~modelado["target_disponible"]
ultimo_mes = modelado["fecha"] == modelado["fecha"].max()

print("Filas del período de modelado:", len(modelado))
display(modelado["particion"].value_counts(dropna=False))
print("Filas sin variable objetivo en el último mes (inferencia):", (sin_target & ultimo_mes).sum())
print("Filas sin variable objetivo por un mes faltante en la serie:", (sin_target & ~ultimo_mes).sum())

assert modelado.loc[modelado["target_disponible"], "particion"].notna().all(), "Hay filas con objetivo sin partición"
assert modelado.loc[sin_target, "particion"].isna().all(), "Hay filas sin objetivo asignadas a una partición"

Filas del período de modelado: 88610


particion
train         49452
test          19737
validation    14320
<NA>           5101
Name: count, dtype: int64

Filas sin variable objetivo en el último mes (inferencia): 5062
Filas sin variable objetivo por un mes faltante en la serie: 39


Las filas sin variable objetivo no se usan para entrenar ni evaluar:

- las del último mes (julio de 2026) son las que se usarían para predecir el mes siguiente, por lo que las exportamos en un dataset de inferencia;
- las demás corresponden a meses seguidos por un hueco en la serie del pozo y se descartan de los datasets de modelado.

In [18]:
train = modelado[modelado["particion"] == "train"].copy()
validation = modelado[modelado["particion"] == "validation"].copy()
test = modelado[modelado["particion"] == "test"].copy()
inferencia = modelado[sin_target & ultimo_mes].copy()

particiones = {"train": train, "validation": validation, "test": test}

### 7. Variable objetivo de clasificación

#### 7.1. Distribución de la variable objetivo en entrenamiento

Para definir las clases usamos únicamente el conjunto de entrenamiento. Primero revisamos la distribución de la producción del mes siguiente.

In [19]:
y_train = train["prod_pet_mes_siguiente"]

print(f"Registros de entrenamiento: {len(y_train):,}")
print(f"Producción igual a 0: {(y_train == 0).mean() * 100:.1f}%")
print(f"Producción menor a 1 m³: {(y_train < 1).mean() * 100:.1f}%")

print("Percentiles de la producción del mes siguiente (m³):")
display(y_train.quantile([0.10, 0.25, 0.33, 0.50, 0.66, 0.75, 0.90, 0.99]).round(2))

Registros de entrenamiento: 49,452
Producción igual a 0: 32.1%
Producción menor a 1 m³: 37.0%
Percentiles de la producción del mes siguiente (m³):


0.10       0.00
0.25       0.00
0.33       0.09
0.50      12.50
0.66     173.76
0.75     534.31
0.90    1768.62
0.99    6057.91
Name: prod_pet_mes_siguiente, dtype: float64

Una primera opción era dividir la producción en tres partes iguales con los percentiles 33 y 66. Sin embargo, como incluimos todos los tipos de pozo, alrededor de un tercio de los registros tiene producción igual a 0: son principalmente pozos gasíferos y de otros tipos. En consecuencia, el percentil 33 queda prácticamente en 0 y la clase "baja" estaría formada casi solo por ceros, con un límite sin significado práctico.

Por eso definimos las clases combinando un criterio de dominio y uno basado en los datos:

| Clase | Definición | Criterio |
|---|---|---|
| Baja | menos de 1 m³ en el mes | Sin producción significativa de petróleo. Incluye los meses sin producción y las trazas de condensado de los pozos de gas (1 m³ al mes equivale a unos 0,03 m³ por día). |
| Media | desde 1 m³ hasta la mediana | La mitad inferior de los registros con producción significativa. |
| Alta | desde la mediana | La mitad superior de los registros con producción significativa. |

La mediana se calcula solo con los registros de entrenamiento con producción significativa, y los mismos límites se aplican sin cambios a validación y test.

In [20]:
umbral_baja = 1.0
umbral_alta = y_train[y_train >= umbral_baja].median()

etiquetas_clase = ["Baja", "Media", "Alta"]


def clasificar_produccion(produccion):
    return pd.cut(
        produccion,
        bins=[-np.inf, umbral_baja, umbral_alta, np.inf],
        right=False,
        labels=etiquetas_clase,
    )


for dataframe in [train, validation, test]:
    dataframe["clase_prod_pet_mes_siguiente"] = clasificar_produccion(dataframe["prod_pet_mes_siguiente"])

print(f"Límite entre Baja y Media: {umbral_baja} m³")
print(f"Límite entre Media y Alta: {umbral_alta:.2f} m³ (aproximadamente {umbral_alta / 30.4:.1f} m³ por día)")

Límite entre Baja y Media: 1.0 m³
Límite entre Media y Alta: 265.75 m³ (aproximadamente 8.7 m³ por día)


#### 7.2. Distribución de las clases

In [21]:
distribucion_clases = pd.DataFrame({
    nombre: dataframe["clase_prod_pet_mes_siguiente"].value_counts(normalize=True).reindex(etiquetas_clase) * 100
    for nombre, dataframe in particiones.items()
}).round(1)

print("Porcentaje de registros por clase:")
display(distribucion_clases)

print("Producción del mes siguiente por clase (entrenamiento):")
display(train.groupby("clase_prod_pet_mes_siguiente", observed=True)["prod_pet_mes_siguiente"].describe().round(1))

Porcentaje de registros por clase:


,train,validation,test
clase_prod_pet_mes_siguiente,,,
Baja,37.0,35.3,35.0
Media,31.5,30.8,29.5
Alta,31.5,33.9,35.5


Producción del mes siguiente por clase (entrenamiento):


,count,mean,std,min,25%,50%,75%,max
clase_prod_pet_mes_siguiente,,,,,,,,
Baja,18316.0,0.1,0.2,0.0,0.0,0.0,0.0,1.0
Media,15568.0,46.8,63.6,1.0,5.7,18.3,55.4,265.7
Alta,15568.0,1684.7,1649.4,265.8,596.0,1111.4,2130.7,20984.5


Las tres clases tienen un tamaño similar en entrenamiento, y su proporción se mantiene en validación y test, aunque los límites se calcularon solo con los datos de entrenamiento.

### 8. Valores faltantes, codificación y escalado

#### 8.1. Valores faltantes de las variables predictoras

Revisamos los valores faltantes de cada variable en cada partición y su origen. No eliminamos registros incompletos ni completamos valores en esta etapa.

In [22]:
motivo_faltante = {
    "prod_pet_lag_1": "Historial no disponible: primeros meses del pozo o mes faltante",
    "prod_pet_lag_2": "Historial no disponible: primeros meses del pozo o mes faltante",
    "prod_pet_lag_3": "Historial no disponible: primeros meses del pozo o mes faltante",
    "prod_pet_media_3m": "Historial no disponible: ventana incompleta",
    "prod_pet_media_6m": "Historial no disponible: ventana incompleta",
    "prod_pet_desvio_3m": "Historial no disponible: ventana incompleta",
    "prod_pet_cambio_abs": "Historial no disponible: sin mes anterior",
    "prod_pet_cambio_rel": "Estructural: mes anterior sin producción, o sin mes anterior",
    "prod_pet_por_dia": "Estructural: mes sin días efectivos",
    "tef": "Valor inválido en la fuente (notebook 02, sección 2.2)",
    "profundidad": "Valor inválido en la fuente (notebook 02, sección 2.3)",
    "cota": "Pozo sin información en el dataset de pozos",
    "tipopozo": "No informado en la fuente",
    "tipoestado": "No informado en la fuente",
    "tipoextraccion": "No informado en la fuente",
    "sub_tipo_recurso": "No informado en la fuente",
    "clasificacion": "No informado en la fuente",
    "subclasificacion": "No informado en la fuente",
}
for columna in columnas_fractura + ["meses_desde_ultima_fractura"]:
    motivo_faltante[columna] = "Cobertura insuficiente: sin fractura disponible al mes de corte"
motivo_faltante["presion_maxima_psi_max"] = "Cobertura insuficiente o presión no informada"
motivo_faltante["potencia_equipos_fractura_hp_max"] = "Cobertura insuficiente o potencia no informada"

faltantes = pd.DataFrame({
    f"% {nombre}": (dataframe[features].isna().mean() * 100).round(2)
    for nombre, dataframe in particiones.items()
})
faltantes = faltantes[faltantes.sum(axis=1) > 0].sort_values("% train", ascending=False)
faltantes["origen"] = faltantes.index.map(motivo_faltante)

display(faltantes)

assert faltantes["origen"].notna().all(), "Hay variables con faltantes sin origen identificado"

,% train,% validation,% test,origen
prod_pet_cambio_rel,33.13,31.30,30.74,"Estructural: mes anterior sin producción, o si..."
potencia_equipos_fractura_hp_max,29.66,26.56,26.37,Cobertura insuficiente o potencia no informada
presion_maxima_psi_max,29.37,26.28,26.10,Cobertura insuficiente o presión no informada
agua_inyectada_m3_acum,29.35,26.26,26.08,Cobertura insuficiente: sin fractura disponibl...
eventos_fractura_acum,29.35,26.26,26.08,Cobertura insuficiente: sin fractura disponibl...
meses_desde_ultima_fractura,29.35,26.26,26.08,Cobertura insuficiente: sin fractura disponibl...
longitud_rama_horizontal_m_max,29.35,26.26,26.08,Cobertura insuficiente: sin fractura disponibl...
arena_bombeada_tn_acum,29.35,26.26,26.08,Cobertura insuficiente: sin fractura disponibl...
cantidad_fracturas_acum,29.35,26.26,26.08,Cobertura insuficiente: sin fractura disponibl...
tipo_terminacion_ultima,29.35,26.26,26.08,Cobertura insuficiente: sin fractura disponibl...


Los valores faltantes tienen cuatro orígenes:

- **Estructurales:** el valor no existe por definición, como el cambio relativo cuando el mes anterior no tuvo producción o la producción por día cuando el pozo no tuvo días efectivos.
- **Historial no disponible:** los rezagos y las ventanas no pueden calcularse en los primeros meses de un pozo o cuando hay meses faltantes en su serie.
- **Valores inválidos en la fuente:** `tef` y `profundidad`, que reemplazamos por faltantes en la notebook 02 por ser imposibles.
- **Cobertura insuficiente:** las variables de fracturación faltan cuando no había ninguna fractura disponible al mes de corte, ya sea porque el pozo no figura en el dataset de fracturación o porque sus fracturas todavía no estaban cargadas. Esto no significa que el pozo no haya sido fracturado, por lo que no corresponde reemplazar estos faltantes por 0. La variable `fractura_previa` indica esta situación de forma explícita.

#### 8.2. Recomendaciones para la etapa de modelado

No ajustamos imputadores, codificadores ni escaladores en esta notebook. Si se ajustaran con todo el dataset, usarían información de validación y test (por ejemplo, la media de una variable en 2026), lo que constituye una fuga de información. Recomendamos aplicar estas transformaciones dentro de un `Pipeline` de scikit-learn, que las ajusta solo con los datos de entrenamiento y las aplica sin cambios a validación y test:

| Transformación | Variables | Recomendación |
|---|---|---|
| Imputación de numéricas | `features_numericas` | Los modelos basados en árboles de gradiente (`HistGradientBoostingRegressor` y `HistGradientBoostingClassifier`) manejan faltantes de forma nativa. Para modelos lineales o de distancia, usar `SimpleImputer(strategy="median", add_indicator=True)`, que además agrega una columna que indica qué valores faltaban. |
| Codificación de categóricas | `features_categoricas` | `OneHotEncoder(handle_unknown="ignore", min_frequency=...)`. `handle_unknown` evita errores con categorías que aparezcan solo en validación o test, y `min_frequency` agrupa las categorías poco frecuentes, por ejemplo en `empresa`. Los faltantes de las categóricas pueden tratarse como una categoría más. |
| Escalado | `features_numericas` | `StandardScaler` solo para modelos lineales o de distancia. Los modelos basados en árboles no lo necesitan. |
| Variables booleanas | `features_booleanas` | Pueden usarse directamente como 0 y 1. |

Además, la producción y la mayoría de las variables de volumen tienen distribuciones muy asimétricas (sección 6 de la notebook 01). Para los modelos lineales conviene evaluar una transformación logarítmica (`np.log1p`), tanto de las variables como de la variable objetivo de regresión.

### 9. Auditoría de fuga de información

#### 9.1. Revisión de cada variable

Para cada variable predictora revisamos qué representa, cuándo está disponible y si usa información posterior al mes de corte.

| Variable | Qué representa | Disponible al cierre del mes t | ¿Usa información posterior a t? | Decisión |
|---|---|---|---|---|
| `prod_pet` | Producción de petróleo del mes t | Sí | No | Incluida |
| `prod_pet_lag_1` a `prod_pet_lag_3` | Producción de los meses t-1 a t-3 | Sí | No (validado en la sección 3.3) | Incluida |
| `prod_pet_media_3m`, `prod_pet_media_6m`, `prod_pet_desvio_3m` | Ventanas que terminan en t | Sí | No: la ventana termina en t (validado en la sección 3.3) | Incluida |
| `prod_pet_cambio_abs`, `prod_pet_cambio_rel` | Cambio entre t-1 y t | Sí | No | Incluida |
| `prod_pet_acumulada` | Suma de la producción hasta t | Sí | No: suma acumulada en orden cronológico | Incluida |
| `prod_pet_por_dia` | Producción de t dividida por los días efectivos de t | Sí | No | Incluida |
| `prod_gas`, `prod_agua`, `tef` | Producción y días efectivos del mes t | Sí | No. El `tef` de t + 1 no forma parte de la tabla | Incluida |
| `tipopozo`, `tipoestado`, `tipoextraccion`, `empresa` | Situación informada para el mes t | Sí | No: se usan los valores mensuales, no el estado actual del pozo (notebook 02, sección 3.3) | Incluida |
| Características fijas del pozo | Ubicación, geología y profundidad | Sí | No: no cambian en el tiempo (notebook 02, sección 2.8) | Incluida |
| `mes`, `meses_desde_inicio`, `inicio_censurado` | Posición en el calendario y antigüedad | Sí | No: el primer registro es anterior a t | Incluida |
| Fracturación por fecha de disponibilidad | Fracturas cargadas hasta t | Sí | No (validado en la sección 4.2) | Incluida |
| Fracturación por fecha del evento (`*_evento`) | Fracturas ocurridas hasta t | No siempre: una parte se cargó después de t | Sí, en una minoría de filas | Excluida; conservada para análisis de sensibilidad |
| `tiene_datos_fractura` | El pozo figura en el dataset de fracturación | No siempre | Sí: depende de cargas posteriores | Excluida |
| `fecha_primera_fractura`, `fecha_ultima_fractura` | Fechas de fracturas | Sí, en su versión disponible | No, pero son fechas absolutas | Excluidas; se usan a través de `meses_desde_ultima_fractura` |
| `prod_pet_mes_siguiente` | Producción de t + 1 | No | Es el objetivo | Excluida (target de regresión) |
| `clase_prod_pet_mes_siguiente` | Clase de la producción de t + 1 | No | Es el objetivo | Excluida (target de clasificación) |
| `target_disponible` | Existencia del registro de t + 1 | No | Sí: depende del mes t + 1 | Excluida |
| `anio` | Año calendario | Sí | No, pero sus valores de test no aparecen en entrenamiento | Excluida |
| `idpozo`, `sigla` | Identificadores | Sí | No, pero permiten memorizar pozos | Excluidas |
| `provincia`, `areapermisoconcesion`, `areayacimiento` | Ubicación administrativa | Sí | No, pero son redundantes o de alta cardinalidad | Excluidas |
| `iny_agua`, `iny_gas` | Inyección del mes t | Sí | No, pero son casi constantes | Excluidas |
| `prod_corregida`, `tef_corregido`, `periodo_modelado`, `meses_desde_registro_anterior` | Control de calidad y estructura de la serie | Sí | No, pero no describen el pozo | Excluidas |

Otras dos fuentes de fuga revisadas:

- **Límites de las clases:** se calcularon solo con el entrenamiento (sección 7.1).
- **Transformaciones calculadas con todo el dataset:** no ajustamos imputadores, codificadores ni escaladores (sección 8.2). Las únicas variables calculadas sobre toda la tabla son las de historial y fracturación, y cada una usa solo información del mismo pozo hasta el mes de corte.

#### 9.2. Validaciones automáticas

Verificamos con `assert` las condiciones que deben cumplir los datasets.

In [23]:
targets = ["prod_pet_mes_siguiente", "clase_prod_pet_mes_siguiente"]
claves = ["idpozo", "fecha"]

claves_por_particion = {
    nombre: set(map(tuple, dataframe[claves].to_numpy())) for nombre, dataframe in particiones.items()
}

validaciones = {}

validaciones["Sin targets entre las variables predictoras"] = not set(targets) & set(features)
validaciones["Sin variables de control o identificadores entre las predictoras"] = not (
    {"target_disponible", "idpozo", "sigla", "fecha", "mes_objetivo", "particion", "anio", "tiene_datos_fractura"} & set(features)
)
validaciones["Sin variables de fracturación por fecha del evento entre las predictoras"] = not set(features_sensibilidad) & set(features)

for nombre, dataframe in particiones.items():
    validaciones[f"{nombre}: una sola fila por pozo y mes"] = not dataframe.duplicated(claves).any()
    validaciones[f"{nombre}: ordenado por pozo y mes"] = dataframe.equals(dataframe.sort_values(claves))
    validaciones[f"{nombre}: target sin faltantes"] = dataframe["prod_pet_mes_siguiente"].notna().all()
    validaciones[f"{nombre}: clase sin faltantes"] = dataframe["clase_prod_pet_mes_siguiente"].notna().all()

validaciones["Sin claves compartidas entre train y validation"] = not claves_por_particion["train"] & claves_por_particion["validation"]
validaciones["Sin claves compartidas entre train y test"] = not claves_por_particion["train"] & claves_por_particion["test"]
validaciones["Sin claves compartidas entre validation y test"] = not claves_por_particion["validation"] & claves_por_particion["test"]

validaciones["Objetivos de train anteriores a los de validation"] = train["mes_objetivo"].max() < validation["mes_objetivo"].min()
validaciones["Objetivos de validation anteriores a los de test"] = validation["mes_objetivo"].max() < test["mes_objetivo"].min()
validaciones["Ningún mes de corte de train posterior a sus objetivos"] = train["fecha"].max() < validation["mes_objetivo"].min()

validaciones["Fracturas disponibles al mes de corte"] = all(
    (dataframe["mes_disponible"].dropna() <= dataframe.loc[dataframe["mes_disponible"].notna(), "fecha"]).all()
    for dataframe in particiones.values()
)
validaciones["Variables de historial validadas (sección 3.3)"] = resultado_historial.all()

resultado_validaciones = pd.Series(validaciones, name="cumple")
display(resultado_validaciones)

assert resultado_validaciones.all(), "Hay validaciones que no se cumplen"

Sin targets entre las variables predictoras                                 True
Sin variables de control o identificadores entre las predictoras            True
Sin variables de fracturación por fecha del evento entre las predictoras    True
train: una sola fila por pozo y mes                                         True
train: ordenado por pozo y mes                                              True
train: target sin faltantes                                                 True
train: clase sin faltantes                                                  True
validation: una sola fila por pozo y mes                                    True
validation: ordenado por pozo y mes                                         True
validation: target sin faltantes                                            True
validation: clase sin faltantes                                             True
test: una sola fila por pozo y mes                                          True
test: ordenado por pozo y me

Como control adicional, verificamos que ninguna variable numérica coincida con el target en casi todos los registros, lo que indicaría que contiene la producción del mes siguiente.

In [24]:
coincidencia_con_target = pd.Series({
    columna: np.isclose(train[columna], train["prod_pet_mes_siguiente"]).mean() * 100
    for columna in features_numericas
}).sort_values(ascending=False)

print("Porcentaje de registros de entrenamiento donde la variable coincide con el target:")
display(coincidencia_con_target.head(5).round(1))

assert coincidencia_con_target.max() < 90, "Alguna variable coincide con el target en casi todos los registros"

Porcentaje de registros de entrenamiento donde la variable coincide con el target:


prod_pet               30.9
prod_pet_lag_1         30.0
prod_pet_cambio_abs    29.8
prod_pet_lag_2         29.4
prod_pet_media_3m      29.0
dtype: float64

Las mayores coincidencias corresponden a la producción del mes de corte y a las variables derivadas de ella, y se deben a los pozos que no producen petróleo en meses consecutivos: en esos casos tanto la variable como el target valen 0. Ninguna variable reproduce el target.

### 10. Exportación de los datasets

#### 10.1. Datasets de modelado

Cada dataset incluye:

- las columnas de identificación y auditoría (`idpozo`, `sigla`, `fecha`, `mes_objetivo` y `particion`), que no deben usarse como variables;
- las variables predictoras;
- las variables de fracturación por fecha del evento, para el análisis de sensibilidad;
- los dos targets.

El dataset de inferencia contiene los registros de julio de 2026, sin target.

In [25]:
columnas_identificacion = ["idpozo", "sigla", "fecha", "mes_objetivo", "particion"]
columnas_exportar = columnas_identificacion + features + features_sensibilidad + targets

archivos_salida = {
    "train_modelado.csv": train,
    "validation_modelado.csv": validation,
    "test_modelado.csv": test,
}

for nombre_archivo, dataframe in archivos_salida.items():
    ruta = ruta_processed / nombre_archivo
    dataframe[columnas_exportar].to_csv(ruta, index=False, date_format="%Y-%m-%d")
    print(f"Guardado: {ruta} | Filas: {len(dataframe):,} | Columnas: {len(columnas_exportar)}")

columnas_inferencia = [columna for columna in columnas_exportar if columna not in targets]
ruta = ruta_processed / "inferencia_modelado.csv"
inferencia[columnas_inferencia].to_csv(ruta, index=False, date_format="%Y-%m-%d")
print(f"Guardado: {ruta} | Filas: {len(inferencia):,} | Columnas: {len(columnas_inferencia)}")

Guardado: ../data/processed/train_modelado.csv | Filas: 49,452 | Columnas: 58


Guardado: ../data/processed/validation_modelado.csv | Filas: 14,320 | Columnas: 58


Guardado: ../data/processed/test_modelado.csv | Filas: 19,737 | Columnas: 58
Guardado: ../data/processed/inferencia_modelado.csv | Filas: 5,062 | Columnas: 56


#### 10.2. Definición de columnas

Exportamos también las listas de variables, los límites de las clases y de las particiones en `columnas_modelado.json`. Las notebooks de modelado pueden leer este archivo para usar exactamente las mismas definiciones.

In [26]:
definicion_columnas = {
    "identificacion": columnas_identificacion,
    "features_numericas": features_numericas,
    "features_booleanas": features_booleanas,
    "features_categoricas": features_categoricas,
    "features_sensibilidad": features_sensibilidad,
    "target_regresion": "prod_pet_mes_siguiente",
    "target_clasificacion": "clase_prod_pet_mes_siguiente",
    "clases": {
        "etiquetas": etiquetas_clase,
        "umbral_baja_media_m3": umbral_baja,
        "umbral_media_alta_m3": round(float(umbral_alta), 4),
    },
    "particiones_por_mes_objetivo": limites_particion,
}

ruta = ruta_processed / "columnas_modelado.json"
with open(ruta, "w", encoding="utf-8") as archivo:
    json.dump(definicion_columnas, archivo, ensure_ascii=False, indent=2)

print(f"Guardado: {ruta}")

Guardado: ../data/processed/columnas_modelado.json


Verificamos que los archivos exportados se lean correctamente y coincidan con los datos en memoria.

In [27]:
for nombre_archivo, dataframe in archivos_salida.items():
    leido = pd.read_csv(ruta_processed / nombre_archivo, parse_dates=["fecha", "mes_objetivo"], low_memory=False)
    print(
        f"{nombre_archivo}: misma forma {leido.shape == dataframe[columnas_exportar].shape} | "
        f"mismas claves {leido[claves].equals(dataframe[claves].reset_index(drop=True))} | "
        f"mismo target {np.allclose(leido['prod_pet_mes_siguiente'], dataframe['prod_pet_mes_siguiente'])} | "
        f"misma clase {(leido['clase_prod_pet_mes_siguiente'] == dataframe['clase_prod_pet_mes_siguiente'].astype(str).to_numpy()).all()}"
    )

train_modelado.csv: misma forma True | mismas claves True | mismo target True | misma clase True
validation_modelado.csv: misma forma True | mismas claves True | mismo target True | misma clase True
test_modelado.csv: misma forma True | mismas claves True | mismo target True | misma clase True


### 11. Resumen

#### 11.1. Resumen de las particiones

In [28]:
def resumen_particion(dataframe):
    y = dataframe["prod_pet_mes_siguiente"]
    return {
        "filas": len(dataframe),
        "pozos": dataframe["idpozo"].nunique(),
        "meses de corte": f"{dataframe['fecha'].min():%Y-%m} a {dataframe['fecha'].max():%Y-%m}",
        "meses del objetivo": f"{dataframe['mes_objetivo'].min():%Y-%m} a {dataframe['mes_objetivo'].max():%Y-%m}",
        "target medio (m³)": round(y.mean(), 1),
        "target mediano (m³)": round(y.median(), 1),
        "% target = 0": round((y == 0).mean() * 100, 1),
        "% clase Baja": round((dataframe["clase_prod_pet_mes_siguiente"] == "Baja").mean() * 100, 1),
        "% clase Media": round((dataframe["clase_prod_pet_mes_siguiente"] == "Media").mean() * 100, 1),
        "% clase Alta": round((dataframe["clase_prod_pet_mes_siguiente"] == "Alta").mean() * 100, 1),
        "% filas con algún faltante": round(dataframe[features].isna().any(axis=1).mean() * 100, 1),
    }


resumen_particiones = pd.DataFrame({nombre: resumen_particion(dataframe) for nombre, dataframe in particiones.items()})
display(resumen_particiones)

total = sum(len(dataframe) for dataframe in particiones.values())
for nombre, dataframe in particiones.items():
    print(f"{nombre}: {len(dataframe) / total * 100:.1f}% de las filas")

pozos_train = set(train["idpozo"])
primer_registro = obt.groupby("idpozo")["fecha"].min()
inicio_validation = validation["fecha"].min()

print()
for nombre, dataframe in [("validation", validation), ("test", test)]:
    pozos_nuevos = list(set(dataframe["idpozo"]) - pozos_train)
    comenzaron_despues = (primer_registro[pozos_nuevos] >= inicio_validation).sum()
    print(
        f"Pozos de {nombre} que no están en train: {len(pozos_nuevos)} "
        f"(con primer registro desde {inicio_validation:%Y-%m}: {comenzaron_despues})"
    )

,train,validation,test
filas,49452,14320,19737
pozos,4707,4814,5015
meses de corte,2025-01 a 2025-11,2025-12 a 2026-02,2026-03 a 2026-06
meses del objetivo,2025-02 a 2025-12,2026-01 a 2026-03,2026-04 a 2026-07
target medio (m³),545.1,596.5,615.7
target mediano (m³),12.5,15.7,18.1
% target = 0,32.1,30.1,29.1
% clase Baja,37.0,35.3,35.0
% clase Media,31.5,30.8,29.5
% clase Alta,31.5,33.9,35.5


train: 59.2% de las filas
validation: 17.1% de las filas
test: 23.6% de las filas

Pozos de validation que no están en train: 114 (con primer registro desde 2025-12: 114)
Pozos de test que no están en train: 316 (con primer registro desde 2025-12: 316)


La mayoría de los pozos de validación y test también aparece en entrenamiento, en meses anteriores. Esto es esperable en una partición temporal: el objetivo es predecir la producción futura de pozos existentes, y ningún mes de un pozo se repite entre particiones.

Los pozos de validación y test que no están en entrenamiento comenzaron a producir después del período de entrenamiento. Para ellos el modelo cuenta con poco historial, por lo que conviene revisar su desempeño por separado.

#### 11.2. Diccionario de variables predictoras

| Variable | Significado | Fuente | Tipo | Disponibilidad | Faltantes |
|---|---|---|---|---|---|
| `prod_pet` | Producción de petróleo del mes de corte (m³) | Producción | numérica | Cierre del mes t | Sin faltantes |
| `prod_pet_lag_1`, `prod_pet_lag_2`, `prod_pet_lag_3` | Producción de 1, 2 y 3 meses antes (m³) | Calculada (grilla de calendario) | numérica | Cierre del mes t | Primeros meses del pozo o mes faltante |
| `prod_pet_media_3m`, `prod_pet_media_6m` | Promedio de los últimos 3 y 6 meses, incluyendo t (m³) | Calculada (grilla de calendario) | numérica | Cierre del mes t | Ventana incompleta |
| `prod_pet_desvio_3m` | Desvío estándar de los últimos 3 meses (m³) | Calculada (grilla de calendario) | numérica | Cierre del mes t | Ventana incompleta |
| `prod_pet_cambio_abs` | Producción de t menos la de t-1 (m³) | Calculada (grilla de calendario) | numérica | Cierre del mes t | Sin mes anterior |
| `prod_pet_cambio_rel` | Cambio relativo entre t-1 y t | Calculada (grilla de calendario) | numérica | Cierre del mes t | Sin mes anterior o mes anterior sin producción |
| `prod_pet_acumulada` | Producción acumulada hasta t (m³) | Calculada | numérica | Cierre del mes t | Sin faltantes; subestimada en pozos con inicio censurado |
| `prod_pet_por_dia` | Producción de t por día efectivo (m³/día) | Calculada | numérica | Cierre del mes t | Meses sin días efectivos |
| `prod_gas` | Producción de gas del mes (miles de m³) | Producción | numérica | Cierre del mes t | Sin faltantes |
| `prod_agua` | Producción de agua del mes (m³) | Producción | numérica | Cierre del mes t | Sin faltantes |
| `tef` | Días efectivos de producción del mes | Producción | numérica | Cierre del mes t | Valores inválidos en la fuente |
| `profundidad` | Profundidad del pozo (m) | Producción | numérica | Fija | Valores inválidos en la fuente |
| `cota` | Altura del terreno en la boca del pozo (m) | Pozos | numérica | Fija | Pozos sin información |
| `coordenadax`, `coordenaday` | Longitud y latitud (grados decimales) | Producción | numérica | Fija | Sin faltantes |
| `mes` | Mes calendario del mes de corte (1 a 12) | Producción | numérica | Cierre del mes t | Sin faltantes |
| `meses_desde_inicio` | Meses desde el primer registro del pozo | Notebook 02 | numérica | Cierre del mes t | Sin faltantes; subestimada en pozos con inicio censurado |
| `inicio_censurado` | El pozo ya producía en enero de 2006 | Notebook 02 | booleana | Fija | Sin faltantes |
| `fractura_previa` | Había alguna fractura cargada al mes de corte | Calculada (fracturación) | booleana | Cierre del mes t | Sin faltantes |
| `eventos_fractura_acum` | Eventos de fracturación cargados hasta t | Calculada (fracturación) | numérica | Cierre del mes t | Sin fractura disponible |
| `cantidad_fracturas_acum` | Fracturas (etapas) cargadas hasta t | Calculada (fracturación) | numérica | Cierre del mes t | Sin fractura disponible |
| `agua_inyectada_m3_acum` | Agua inyectada en las fracturas cargadas hasta t (m³) | Calculada (fracturación) | numérica | Cierre del mes t | Sin fractura disponible |
| `arena_bombeada_tn_acum` | Arena bombeada en las fracturas cargadas hasta t (toneladas) | Calculada (fracturación) | numérica | Cierre del mes t | Sin fractura disponible |
| `longitud_rama_horizontal_m_max` | Mayor longitud de rama horizontal cargada hasta t (m) | Calculada (fracturación) | numérica | Cierre del mes t | Sin fractura disponible |
| `presion_maxima_psi_max` | Mayor presión máxima cargada hasta t (psi) | Calculada (fracturación) | numérica | Cierre del mes t | Sin fractura disponible o presión no informada |
| `potencia_equipos_fractura_hp_max` | Mayor potencia de equipos cargada hasta t (hp) | Calculada (fracturación) | numérica | Cierre del mes t | Sin fractura disponible o potencia no informada |
| `meses_desde_ultima_fractura` | Meses entre la fractura más reciente cargada y t | Calculada (fracturación) | numérica | Cierre del mes t | Sin fractura disponible |
| `tipopozo` | Tipo de pozo informado en el mes | Producción | categórica | Cierre del mes t | Pocos registros no informados |
| `tipoestado` | Estado del pozo informado en el mes | Producción | categórica | Cierre del mes t | Pocos registros no informados |
| `tipoextraccion` | Sistema de extracción informado en el mes | Producción | categórica | Cierre del mes t | Pocos registros no informados |
| `empresa` | Empresa operadora en el mes | Producción | categórica | Cierre del mes t | Sin faltantes |
| `cuenca` | Cuenca del pozo | Producción | categórica | Fija | Sin faltantes |
| `formacion` | Formación geológica productiva | Producción | categórica | Fija | Sin faltantes |
| `sub_tipo_recurso` | SHALE o TIGHT | Producción | categórica | Fija | Pocos registros no informados |
| `clasificacion` | Explotación, exploración o servicio | Producción | categórica | Fija | Pocos registros no informados |
| `subclasificacion` | Desarrollo, avanzada, etc. | Producción | categórica | Fija | Pocos registros no informados |
| `proyecto` | GAS PLUS o Sin Proyecto | Producción | categórica | Fija | Sin faltantes |
| `tipo_terminacion_ultima` | Tipo de terminación de la fractura más reciente cargada | Calculada (fracturación) | categórica | Cierre del mes t | Sin fractura disponible |

**Targets**

| Variable | Significado | Tipo |
|---|---|---|
| `prod_pet_mes_siguiente` | Producción de petróleo del mes t + 1 (m³) | Regresión |
| `clase_prod_pet_mes_siguiente` | Baja (menos de 1 m³), Media (desde 1 m³ hasta la mediana de entrenamiento) o Alta (desde la mediana) | Clasificación |

#### 11.3. Limitaciones

- **Datos disponibles para entrenar.** El período de modelado tiene 18 meses con target. Con la partición elegida, el entrenamiento cubre 11 meses, por lo que el modelo puede no capturar comportamientos que se repiten a lo largo de varios años. Si se quisiera más historia, podría ampliarse el entrenamiento a años anteriores, reconsiderando la decisión de la notebook 01.
- **Fecha de carga como disponibilidad.** Asumimos que una fractura está disponible desde el mes de su carga. El dataset es de actualización diaria y no informa si los registros se modificaron después de su primera carga.
- **Fracturación incompleta.** Una parte de los pozos no tiene fracturas disponibles. Sus variables de fracturación son faltantes y no ceros.
- **Antigüedad de los pozos más antiguos.** `meses_desde_inicio` y `prod_pet_acumulada` subestiman la antigüedad y la producción acumulada de los pozos con inicio censurado.
- **Clases y tipos de pozo.** La clase Baja está formada principalmente por pozos que no son petrolíferos. Al evaluar el clasificador conviene revisar también su desempeño separado por `tipopozo`, para verificar que distinga niveles de producción y no solo el tipo de pozo.
- **Rezagos sin historial.** Los pozos nuevos tienen faltantes en sus rezagos y ventanas durante sus primeros meses. El modelo elegido debe poder manejar estos faltantes (sección 8.2).